# Phase 10 — Final Report Assembly

**Project:** Wholesale Customer Segmentation

Phase 10 - Submission-ready report assembly.

Builds a JSON manifest from the actual source data and pipeline outputs. It is
intentionally self-contained: it does not require a separately generated
Markdown section, so the full pipeline can be rebuilt from a clean directory.

### How to use this notebook
Run cells from top to bottom. Keep the project files in the same folder as this notebook. Phases 1–4 create the data preparation artifacts used by later phases; Phases 5–9 read those artifacts; Phase 10 assembles the final report.

In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import f_oneway
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler



# Recreate upstream constants locally. This notebook is intentionally self-contained.
DATA_PATH = "data/raw/wholesale_customers.csv"
SPEND_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

RANDOM_STATE = 42
N_INIT = 20
PRIMARY_CLUSTER_COL = "cluster_k2"
STABILITY_KS = [2, 3, 4]
SEEDS = [42, 7, 123, 2024, 99]
N_BOOTSTRAP = 50
STABILITY_THRESHOLD = 0.75
UCI_URL = "https://archive.ics.uci.edu/dataset/292/wholesale+customers"


def load_sources() -> dict:
    """Load source and labeled data without depending on intermediate prose files."""
    df_raw = pd.read_csv(DATA_PATH)
    df_labeled = pd.read_csv("labeled_customers.csv")
    if len(df_raw) != len(df_labeled):
        raise ValueError("Raw and labeled datasets have different row counts.")
    return {"df_raw": df_raw, "df_labeled": df_labeled}


def compute_data_overview(df: pd.DataFrame) -> dict:
    return {
        "shape": df.shape,
        "missing_total": int(df.isna().sum().sum()),
        "duplicates": int(df.duplicated().sum()),
        "channel_counts": df["Channel"].value_counts().sort_index().to_dict(),
        "region_counts": df["Region"].value_counts().sort_index().to_dict(),
    }


def compute_vif(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    X = df[cols].values
    rows = []
    for i, col in enumerate(cols):
        y = X[:, i]
        others = np.delete(X, i, axis=1)
        reg = LinearRegression().fit(others, y)
        r2 = reg.score(others, y)
        vif = np.inf if np.isclose(r2, 1.0) else 1.0 / (1.0 - r2)
        rows.append({"feature": col, "VIF": vif})
    return pd.DataFrame(rows).set_index("feature")


def compute_preprocessing(df: pd.DataFrame) -> dict:
    X_raw = df[SPEND_COLS].copy()
    skew_before = X_raw.skew()
    outlier_counts = {}
    for col in SPEND_COLS:
        q1, q3 = X_raw[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        mask = (X_raw[col] < q1 - 1.5 * iqr) | (X_raw[col] > q3 + 1.5 * iqr)
        outlier_counts[col] = int(mask.sum())
    vif = compute_vif(df, SPEND_COLS)
    log_df = np.log1p(X_raw)
    skew_after = log_df.skew()
    scaler = StandardScaler()
    scaled = scaler.fit_transform(log_df)
    scaled_df = pd.DataFrame(scaled, columns=SPEND_COLS)
    return {
        "skew_before": skew_before.round(2),
        "skew_after": skew_after.round(2),
        "outlier_counts": outlier_counts,
        "vif": vif.round(2),
        "scaled_df": scaled_df,
    }


def compute_k_selection(X: np.ndarray) -> dict:
    rows = []
    for k in range(2, 11):
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
        km_labels = km.fit_predict(X)
        agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
        agg_labels = agg.fit_predict(X)
        rows.append({
            "k": k,
            "kmeans_inertia": float(km.inertia_),
            "kmeans_silhouette": float(silhouette_score(X, km_labels)),
            "hier_silhouette": float(silhouette_score(X, agg_labels)),
        })
    table = pd.DataFrame(rows)
    elbow_idx = int(np.argmin(np.diff(np.diff(table["kmeans_inertia"].values))) + 2)
    elbow_k = int(table.iloc[elbow_idx]["k"]) if 0 <= elbow_idx < len(table) else None
    best_k = int(table.loc[table["kmeans_silhouette"].idxmax(), "k"])
    best_hier = int(table.loc[table["hier_silhouette"].idxmax(), "k"])
    return {"table": table, "best_k": best_k, "best_hier": best_hier, "elbow_k": elbow_k}


def seed_stability(X: np.ndarray, k: int) -> float:
    labels = [KMeans(n_clusters=k, random_state=s, n_init=N_INIT).fit_predict(X) for s in SEEDS]
    aris = [
        adjusted_rand_score(labels[i], labels[j])
        for i in range(len(labels))
        for j in range(i + 1, len(labels))
    ]
    return float(np.mean(aris))


def bootstrap_stability(X: np.ndarray, k: int) -> float:
    baseline = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT).fit(X)
    baseline_labels = baseline.labels_
    values = []
    n = len(X)
    rng = np.random.RandomState(12345)
    for _ in range(N_BOOTSTRAP):
        boot_idx = rng.choice(np.arange(n), size=n, replace=True)
        oob = np.setdiff1d(np.arange(n), np.unique(boot_idx))
        if len(oob) < max(10, 2 * k):
            continue
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT).fit(X[boot_idx])
        values.append(adjusted_rand_score(baseline_labels[oob], model.predict(X[oob])))
    if not values:
        raise RuntimeError(f"No usable bootstrap samples for K={k}.")
    return float(np.mean(values))


def compute_stability(X: np.ndarray) -> pd.DataFrame:
    rows = []
    for k in STABILITY_KS:
        seed = seed_stability(X, k)
        boot = bootstrap_stability(X, k)
        rows.append({
            "K": k,
            "Seed variation ARI": seed,
            "Bootstrap ARI": boot,
            "Verdict": "Stable" if seed >= STABILITY_THRESHOLD and boot >= STABILITY_THRESHOLD else "Not stable",
        })
    return pd.DataFrame(rows)


def compute_cluster_profiles(df_labeled: pd.DataFrame) -> dict:
    sizes = df_labeled[PRIMARY_CLUSTER_COL].value_counts().sort_index()
    pct = sizes / len(df_labeled) * 100
    means = df_labeled.groupby(PRIMARY_CLUSTER_COL)[SPEND_COLS].mean()
    medians = df_labeled.groupby(PRIMARY_CLUSTER_COL)[SPEND_COLS].median()

    rows = []
    for col in SPEND_COLS:
        groups = [g[col].values for _, g in df_labeled.groupby(PRIMARY_CLUSTER_COL)]
        f_stat, p_val = f_oneway(*groups)
        grand_mean = df_labeled[col].mean()
        ss_between = sum(len(g) * (g[col].mean() - grand_mean) ** 2 for _, g in df_labeled.groupby(PRIMARY_CLUSTER_COL))
        ss_total = ((df_labeled[col] - grand_mean) ** 2).sum()
        eta2 = ss_between / ss_total if ss_total else 0.0
        rows.append({"Feature": col, "F-statistic": float(f_stat), "Eta-squared": float(eta2), "P-value": float(p_val)})
    separation = pd.DataFrame(rows).sort_values("F-statistic", ascending=False)

    channel_pct = pd.crosstab(
        df_labeled[PRIMARY_CLUSTER_COL], df_labeled["Channel"].map({1: "Horeca", 2: "Retail"}), normalize="index"
    ) * 100
    region_pct = pd.crosstab(df_labeled[PRIMARY_CLUSTER_COL], df_labeled["Region"], normalize="index") * 100

    total_spend = means.sum(axis=1)
    return {
        "sizes": sizes,
        "pct": pct,
        "means": means,
        "medians": medians,
        "separation": separation,
        "channel_pct": channel_pct,
        "region_pct": region_pct,
        "total_spend": total_spend,
    }


def compute_pca(scaled_df: pd.DataFrame) -> dict:
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    pcs = pca.fit_transform(scaled_df[SPEND_COLS])
    loadings = pd.DataFrame(pca.components_.T, columns=["PC1", "PC2"], index=SPEND_COLS)
    return {"explained": pca.explained_variance_ratio_, "loadings": loadings, "pcs": pcs}


def extract_snippet(filepath: str, start: str, end: str) -> str:
    text = Path(filepath).read_text()
    i = text.index(start)
    j = text.index(end, i)
    return text[i:j].rstrip()


def curate_code_snippets() -> dict:
    return {
        "load": extract_snippet("phase1_setup.py", "df_raw = pd.read_csv(DATA_PATH)", "def profile_dataset"),
        "transform": extract_snippet("phase4_transform_scale.py", "log_df = features_df.copy()", "print(\"Applied log1p"),
        "scale": extract_snippet("phase4_transform_scale.py", "scaler = StandardScaler()", "print(\"Applied StandardScaler"),
        "k_loop": extract_snippet("phase5_algorithm_selection.py", "for k in K_RANGE:\n", "kmeans_results = pd.DataFrame(rows)"),
        "final": extract_snippet("phase6_train_validate.py", "for k in CANDIDATE_KS:\n", "print(\"\\n[OK] Final model"),
        "stability": extract_snippet("phase6_train_validate.py", "for seed in STABILITY_SEEDS:", "ari_df = pd.DataFrame(rows)"),
        "pca": extract_snippet("phase8_pca_visualization.py", "pca = PCA(n_components=2, random_state=RANDOM_STATE)", "explained_var = pca.explained_variance_ratio_"),
    }


def df_manifest(df: pd.DataFrame, decimals: int = 1) -> dict:
    x = df.copy()
    x = x.round(decimals)
    return {"columns": x.columns.tolist(), "index": [str(i) for i in x.index.tolist()], "data": x.values.tolist()}


def series_dict(s: pd.Series, decimals: int | None = None) -> dict:
    out = {}
    for k, v in s.items():
        value = float(v)
        out[str(k)] = round(value, decimals) if decimals is not None else value
    return out


def build_interpretation(profiles: dict) -> str:
    names = {}
    blocks = []
    means = profiles["means"]
    total = profiles["total_spend"]
    channel = profiles["channel_pct"]
    for cid in means.index:
        top2 = means.loc[cid].sort_values(ascending=False).index[:2].tolist()
        total_label = "higher-total-spend" if total.loc[cid] == total.max() else "lower-total-spend"
        names[cid] = f"{total_label.title()} {top2[0]}/{top2[1]}-Oriented Segment"
        majority_channel = channel.loc[cid].idxmax()
        blocks.append(
            f"Cluster {cid} — {names[cid]}\n"
            f"Size: {profiles['sizes'].loc[cid]} customers ({profiles['pct'].loc[cid]:.1f}%).\n"
            f"Highest mean-spend categories: {top2[0]} ({means.loc[cid, top2[0]]:,.0f}) and {top2[1]} ({means.loc[cid, top2[1]]:,.0f}).\n"
            f"Mean total annual spend: {total.loc[cid]:,.0f}.\n"
            f"Post-hoc Channel profile: {channel.loc[cid, majority_channel]:.1f}% {majority_channel}."
        )
    return "\n\n".join(blocks), names


def main():
    sources = load_sources()
    raw = sources["df_raw"]
    labeled = sources["df_labeled"]
    overview = compute_data_overview(raw)
    prep = compute_preprocessing(raw)
    scaled = prep["scaled_df"]
    ksel = compute_k_selection(scaled.values)
    stability = pd.read_csv("stability_summary.csv")
    if set(STABILITY_KS) != set(stability["K"].astype(int)):
        raise ValueError("stability_summary.csv does not contain K=2, K=3, and K=4.")
    profiles = compute_cluster_profiles(labeled)
    pca = compute_pca(scaled)
    snippets = curate_code_snippets()
    interpretation_text, cluster_names = build_interpretation(profiles)

    best_k = ksel["best_k"]
    if best_k != 2:
        raise RuntimeError(f"This report builder expects K=2 as primary, but recomputation selected K={best_k}.")
    k2_stable = stability.loc[stability["K"].astype(int) == 2].iloc[0]
    k3_stable = stability.loc[stability["K"].astype(int) == 3].iloc[0]
    k4_stable = stability.loc[stability["K"].astype(int) == 4].iloc[0]

    mean_profile = profiles["means"]
    median_profile = profiles["medians"]
    high_cluster = int(profiles["total_spend"].idxmax())
    low_cluster = int(profiles["total_spend"].idxmin())
    top_sep = profiles["separation"].iloc[0]["Feature"]

    manifest = {
        "title": "Wholesale Customer Segmentation: A Clustering Analysis",
        "subtitle": "Unsupervised Learning Applied to the UCI Wholesale Customers Dataset",
        "sections": {
            "introduction": {
                "heading": "1. Introduction & Dataset Overview",
                "objective_text": (
                    "This project applies unsupervised clustering to segment wholesale customers by annual spending "
                    "across six product categories. The objective is to identify purchasing-pattern segments that can "
                    "support business hypotheses around promotions, service design, inventory planning, and customer management. "
                    "K-Means is the primary method, while Ward hierarchical clustering is used as a methodological cross-check."
                ),
                "dataset_source": UCI_URL,
                "data_dictionary": [
                    ["Channel", "Distribution channel: 1 = Horeca, 2 = Retail", "Post-hoc profiling"],
                    ["Region", "Region code: 1 = Lisbon, 2 = Oporto, 3 = Other", "Post-hoc profiling"],
                    ["Fresh", "Annual spending on fresh products", "Clustering"],
                    ["Milk", "Annual spending on milk products", "Clustering"],
                    ["Grocery", "Annual spending on grocery products", "Clustering"],
                    ["Frozen", "Annual spending on frozen products", "Clustering"],
                    ["Detergents_Paper", "Annual spending on detergents and paper", "Clustering"],
                    ["Delicassen", "Annual spending on delicatessen products", "Clustering"],
                ],
            },
            "data_overview": {
                "heading": "2. Data Overview & Quality",
                "shape": overview["shape"],
                "missing_total": overview["missing_total"],
                "duplicates": overview["duplicates"],
                "channel_counts": overview["channel_counts"],
                "region_counts": overview["region_counts"],
                "negative_spend_total": int((raw[SPEND_COLS] < 0).sum().sum()),
                "zero_spend_total": int((raw[SPEND_COLS] == 0).sum().sum()),
                "constant_columns": [c for c in raw.columns if raw[c].nunique() <= 1],
                "figures": [
                    ["figures/01_histograms_spend_categories.png", "Figure 1. Annual spending distributions across the six clustering categories; all are strongly right-skewed."],
                    ["figures/02_boxplots_spend_categories.png", "Figure 2. Boxplots showing spread and potential high-spend observations."],
                    ["figures/03_channel_region_bars.png", "Figure 3. Distribution of Channel and Region categories."],
                    ["figures/04_correlation_heatmap.png", "Figure 4. Correlation structure among the six spending variables."],
                    ["figures/05_pairplot_by_channel.png", "Figure 5. Exploratory pairplot colored by Channel; Channel is not used as a clustering input."],
                ],
            },
            "methodology": {
                "heading": "3. Methodology",
                "subsections": [
                    {
                        "title": "3.1 Feature Selection & Redundancy Diagnostics",
                        "text": (
                            "The six spending variables were used as the clustering feature set because they directly describe purchasing behavior. "
                            "Channel and Region were excluded from clustering and reserved for post-hoc profiling. Correlation and VIF were examined "
                            "as redundancy diagnostics. Because the spending categories are individually business-relevant, no feature was dropped. "
                            "PCA was reserved for visualization rather than used as a substitute for the six-feature K-Means input."
                        ),
                        "vif": df_manifest(prep["vif"].rename(columns={"VIF": "VIF"}), 2),
                    },
                    {
                        "title": "3.2 Skewness, Outliers & Treatment",
                        "text": (
                            "All six spend variables are strongly right-skewed. IQR screening identified potential extreme observations, "
                            "but these were retained because the available evidence does not establish them as data errors and high-spend customers "
                            "are substantively relevant to segmentation. A log1p transformation was applied to compress extreme values and reduce skew. "
                            "The transformation was evaluated by comparing skewness before and after it."
                        ),
                        "skew_before": series_dict(prep["skew_before"], 2),
                        "skew_after": series_dict(prep["skew_after"], 2),
                        "outlier_counts": prep["outlier_counts"],
                        "figures": [
                            ["figures/06_outlier_annotated_boxplots.png", "Figure 6. IQR-based outlier counts by spending category."],
                            ["figures/07_log_transform_before_after.png", "Figure 7. Effect of log1p on the six spending distributions."],
                            ["figures/08_scaled_features_boxplot.png", "Figure 8. Final standardized feature distributions."],
                        ],
                        "code": snippets["transform"],
                        "code_caption": "Code: log1p transformation applied to the six spending variables.",
                    },
                    {
                        "title": "3.3 Scaling",
                        "text": (
                            "After transformation, StandardScaler was applied so each clustering feature contributed on a comparable standardized scale. "
                            "This is important because K-Means minimizes squared Euclidean distances and is therefore sensitive to feature scale."
                        ),
                        "code": snippets["scale"],
                        "code_caption": "Code: StandardScaler fitted to the log-transformed feature matrix.",
                    },
                    {
                        "title": "3.4 Algorithm Selection & Number of Clusters",
                        "text": (
                            f"K-Means and Ward hierarchical clustering were evaluated for K=2 through K=10. "
                            f"K-Means achieved its highest average silhouette at K={best_k} ({ksel['table'].loc[ksel['table']['k']==best_k, 'kmeans_silhouette'].iloc[0]:.4f}). "
                            f"A second-difference elbow heuristic flagged K={ksel['elbow_k']}; the curve is gradual rather than displaying a uniquely sharp elbow, so this diagnostic was treated as supporting evidence only. "
                            f"The hierarchical model's highest silhouette occurred at K={ksel['best_hier']}. The final choice was therefore based on the combination of "
                            "silhouette separation, elbow behavior, cross-method comparison, stability, and business interpretability rather than on a single metric."
                        ),
                        "k_table": df_manifest(ksel["table"].rename(columns={"k": "K", "kmeans_inertia": "K-Means inertia", "kmeans_silhouette": "K-Means silhouette", "hier_silhouette": "Hierarchical silhouette"}), 4),
                        "figures": [
                            ["figures/09_elbow_method.png", "Figure 9. K-Means elbow curve across K=2-10."],
                            ["figures/10_dendrogram.png", "Figure 10. Ward hierarchical dendrogram used as a qualitative structural cross-check."],
                            ["figures/11_silhouette_comparison.png", "Figure 11. Average silhouette scores for K-Means and hierarchical clustering."],
                            ["figures/12_detailed_silhouette_k2.png", "Figure 12. Detailed silhouette profile for the selected K=2 solution."],
                        ],
                        "code": snippets["k_loop"],
                        "code_caption": "Code: K-Means evaluation across K=2-10 using inertia and silhouette score.",
                    },
                    {
                        "title": "3.5 Final Model & Stability Validation",
                        "text": (
                            f"The primary K-Means model uses K=2, random_state={RANDOM_STATE}, and n_init={N_INIT}. "
                            f"Five random seeds and {N_BOOTSTRAP} bootstrap resamples were used to test whether the clustering was sensitive to initialization or sample variation. "
                            f"K=2 had seed-variation ARI={k2_stable['Seed ARI']:.2f} and bootstrap ARI={k2_stable['Bootstrap ARI']:.2f}; both exceed the project-defined heuristic stability threshold of {STABILITY_THRESHOLD:.2f}. "
                            f"K=3 and K=4 were also checked because they are plausible more-granular alternatives. K=3 had seed ARI={k3_stable['Seed ARI']:.2f} and bootstrap ARI={k3_stable['Bootstrap ARI']:.2f}; "
                            f"K=4 had seed ARI={k4_stable['Seed ARI']:.2f} and bootstrap ARI={k4_stable['Bootstrap ARI']:.2f}."
                        ),
                        "stability_table": df_manifest(stability[["K", "Seed ARI", "Bootstrap ARI", "Verdict"]], 2),
                        "figures": [["figures/13_stability_ari_summary.png", "Figure 13. Mean stability scores for K=2, K=3, and K=4."]],
                        "code": snippets["final"],
                        "code_caption": "Code: final K-Means fitting with fixed reproducibility settings.",
                        "code2": snippets["stability"],
                        "code2_caption": "Code: Adjusted Rand Index stability check across random seeds.",
                    },
                ],
            },
            "results": {
                "heading": "4. Results",
                "subsections": [
                    {
                        "title": "4.1 Final Cluster Profiles",
                        "sizes": {str(k): int(v) for k, v in profiles["sizes"].items()},
                        "pct": series_dict(profiles["pct"], 1),
                        "mean_profile": df_manifest(mean_profile, 1),
                        "median_profile": df_manifest(median_profile, 1),
                        "separation": df_manifest(profiles["separation"].set_index("Feature")[["F-statistic", "Eta-squared"]], 3),
                        "channel_ct": df_manifest(profiles["channel_pct"], 1),
                        "region_ct": df_manifest(profiles["region_pct"], 1),
                        "top_separation_feature": str(top_sep),
                        "figures": [
                            ["figures/14_cluster_profile_bars_cluster_k2.png", "Figure 14. Mean annual spend by product category and cluster."],
                            ["figures/16_feature_discriminating_power_cluster_k2.png", "Figure 15. Descriptive ANOVA F-statistics ranking observed feature separation."],
                            ["figures/17_channel_crosstab_cluster_k2.png", "Figure 16. Channel composition within the discovered clusters."],
                            ["figures/18_region_crosstab_cluster_k2.png", "Figure 17. Region composition within the discovered clusters."],
                        ],
                        "code": snippets["final"],
                        "code_caption": "Code: cluster labeling and retention of original, human-readable spend values for profiling.",
                    },
                    {
                        "title": "4.2 PCA Visualization",
                        "explained_variance": pca["explained"].tolist(),
                        "loadings": df_manifest(pca["loadings"], 3),
                        "text": (
                            f"PCA reduces the six-dimensional standardized feature space to two components for visualization only. "
                            f"PC1 explains {pca['explained'][0]*100:.1f}% of the variance and PC2 explains {pca['explained'][1]*100:.1f}%, "
                            f"for a combined {pca['explained'].sum()*100:.1f}%. The PCA plot is a two-dimensional projection and is not the basis for the clustering itself."
                        ),
                        "figures": [
                            ["figures/20_pca_loadings.png", "Figure 18. Feature loadings on PC1 and PC2."],
                            ["figures/21_pca_cluster_scatter.png", "Figure 19. K=2 clusters projected into PCA space."],
                            ["figures/22_pca_cluster_channel_overlay.png", "Figure 20. Cluster membership and Channel shown together in PCA space."],
                        ],
                        "code": snippets["pca"],
                        "code_caption": "Code: PCA used solely to create a two-dimensional diagnostic visualization.",
                    },
                ],
            },
            "cluster_interpretation": {
                "heading": "5. Cluster Interpretation",
                "text": interpretation_text,
                "context": (
                    f"Cluster {high_cluster} has the higher mean total annual spend ({profiles['total_spend'].loc[high_cluster]:,.0f}) and is strongest in "
                    f"{mean_profile.loc[high_cluster].idxmax()} and {mean_profile.loc[high_cluster].sort_values(ascending=False).index[1]}. "
                    f"Cluster {low_cluster} has the lower mean total annual spend ({profiles['total_spend'].loc[low_cluster]:,.0f}) and is strongest in "
                    f"{mean_profile.loc[low_cluster].idxmax()} and {mean_profile.loc[low_cluster].sort_values(ascending=False).index[1]}."
                ),
            },
            "business_implications": {
                "heading": "6. Business Implications",
                "text": (
                    f"Targeted promotions: Cluster {high_cluster}'s {mean_profile.loc[high_cluster].idxmax()} emphasis supports testing category-bundled offers. "
                    f"Cluster {low_cluster}'s Fresh/Frozen-oriented profile supports testing more category-specific offers.\\n\\n"
                    f"Service and inventory planning: Cluster {low_cluster} is strongly Horeca-dominant and Fresh/Frozen-heavy. This supports a hypothesis around "
                    "perishable-focused service options or different delivery frequencies, but delivery frequency is not contained in this dataset and must be validated separately.\\n\\n"
                    f"Commercial strategy: Cluster {high_cluster} has higher mean annual spend ({profiles['total_spend'].loc[high_cluster]:,.0f} vs. "
                    f"{profiles['total_spend'].loc[low_cluster]:,.0f}), supporting a hypothesis for testing volume-based pricing or loyalty structures. "
                    "These are decision-support hypotheses, not causal findings."
                ),
            },
            "limitations": {
                "heading": "7. Limitations",
                "text": (
                    "1. Sample size: 440 customers is adequate for an educational clustering analysis but still limits external generalization.\\n\\n"
                    "2. Time aggregation: the variables are annual spending totals, so seasonality and customer movement between segments cannot be observed.\\n\\n"
                    "3. Feature scope: the dataset lacks order frequency, margin, tenure, business size, and other firmographic variables that could explain the underlying reasons for different spend patterns.\\n\\n"
                    "4. K-Means geometry: K-Means is most naturally suited to compact, roughly spherical clusters under Euclidean distance. The current solution should therefore be treated as one useful segmentation view rather than a unique ground truth.\\n\\n"
                    "5. Log transformation: the model operates in transformed standardized space, so distances in that space should not be translated directly into monetary differences. Raw means and medians are the correct basis for dollar-denominated interpretation.\\n\\n"
                    "6. Coarse segmentation: K=2 is the selected stable solution, but its relatively low silhouette and strong post-hoc Channel alignment indicate that it captures a broad purchasing split and may miss finer segments."
                ),
            },
            "conclusion": {
                "heading": "8. Conclusion",
                "text": (
                    f"The project identified two stable customer segments from six annual spending variables. K=2 was selected because it achieved the highest K-Means silhouette "
                    f"({ksel['table'].loc[ksel['table']['k']==2, 'kmeans_silhouette'].iloc[0]:.3f}) in the tested K range and remained stable under both seed and bootstrap checks. "
                    "The elbow curve suggested a more granular region, but the K=3 and K=4 alternatives did not provide a stronger overall evidence base once stability and separation were considered.\\n\\n"
                    "The two spend-based segments align strongly with the pre-existing Channel variable even though Channel was excluded from clustering. This is post-hoc contextual evidence of business relevance, not external validation. "
                    "The result is therefore best interpreted as a broad segmentation of purchasing behavior rather than a definitive taxonomy of all customer types."
                ),
            },
            "references": {
                "heading": "9. References",
                "items": [
                    ["UCI Machine Learning Repository — Wholesale Customers", UCI_URL],
                    ["Scikit-learn KMeans documentation", "https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html"],
                    ["Scikit-learn silhouette_score documentation", "https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html"],
                ],
            },
            "appendix": {
                "heading": "Appendix: Reproducibility & Full Pipeline Reference",
                "text": (
                    "The analysis is implemented as phases 1 through 10. The final report manifest is recomputed from the source data and current intermediate outputs rather than from copied numbers. "
                    "The pipeline uses K-Means with random_state=42 and n_init=20. The primary clustering features are the six spend variables; Channel and Region are held out for post-hoc profiling. "
                    "All major transformations, K-selection diagnostics, stability checks, cluster profiles, PCA diagnostics, and report-generation inputs are retained as intermediate artifacts."
                ),
            },
        },
    }

    Path("report_content.json").write_text(json.dumps(manifest, indent=2, default=lambda o: o))
    print("[OK] report_content.json written from fresh source-data recomputation.")


if __name__ == "__main__":
    main()

### Phase 10 checkpoint

Review the outputs and figures generated by this phase before moving to the next phase.